# Agent_Structure 프레임워크 튜토리얼

**DeepAgents + LangGraph + LangChain 기반 플러그인형 AI 에이전트 뼈대**

---

## 이 노트북에서 배울 내용

1. 환경 설정과 API 키 관리
2. 모델 프로바이더 (LLM 교체가 쉬운 이유)
3. 도구(Tool) 레지스트리와 플러그인 시스템
4. **첫 번째 에이전트 만들고 실행하기**
5. 모델 교체, 시스템 프롬프트, 도구 필터링
6. 스트리밍 & 비동기 실행
7. 커스텀 도구 만들기
8. 서브에이전트 활용
9. 커스텀 모델 프로바이더 추가
10. 전체 아키텍처 정리
11. 치트시트 & 다음 단계

## 전체 아키텍처



```
┌─────────────────────────────────────────────────────┐
│                   build_agent()                     │
│                                                     │
│  ┌────────────┐  ┌────────────┐  ┌───────────────┐  │
│  │  Provider  │  │   Tools    │  │  Subagents    │  │
│  │  (LLM)     │  │  Registry  │  │  Registry     │  │
│  └─────┬──────┘  └─────┬──────┘  └──────┬────────┘  │
│        │               │                │           │
│        └───────────────┼────────────────┘           │
│                        ▼                            │
│               create_deep_agent()                   │
│                        │                            │
│                        ▼                            │
│              CompiledStateGraph                     │
│           (LangGraph 기반 에이전트)                  │
└─────────────────────────────────────────────────────┘
```



각 구성요소(Provider, Tools, Subagents)는 **독립적으로 개발**하고,  
`build_agent()`가 이를 **한 곳에서 조립**합니다.

## 사전 준비

- Python 3.10+
- `pip install -r requirements.txt`
- `.env` 파일에 API 키 설정 (최소 `ANTHROPIC_API_KEY`)

---
## 1. 환경 설정

`config/settings.py`에 `Settings` dataclass가 정의되어 있습니다.  
`.env` 파일 또는 환경 변수에서 API 키와 기본 설정을 자동으로 로드합니다.

| 환경 변수 | 용도 | 필수 여부 |
|-----------|------|----------|
| `ANTHROPIC_API_KEY` | Anthropic(Claude) 모델 사용 | 기본 프로바이더 사용 시 필수 |
| `OPENAI_API_KEY` | OpenAI 모델 사용 | OpenAI 프로바이더 사용 시 필수 |
| `TAVILY_API_KEY` | 웹 검색 도구 | web_search 사용 시 필수 |
| `DEFAULT_PROVIDER` | 기본 프로바이더 이름 | 선택 (기본: `"anthropic"`) |
| `DEFAULT_MODEL` | 기본 모델명 | 선택 (기본: `"claude-sonnet-4-5-20250929"`) |

In [1]:
import sys, os

# 패키지 상위 디렉토리를 경로에 추가 (노트북이 패키지 내부에 있으므로)
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

In [2]:
from Agent_Structure.config import settings

# 현재 설정 확인
print(f"기본 프로바이더: {settings.default_provider}")
print(f"기본 모델:       {settings.default_model}")
print(f"최대 반복 횟수:  {settings.max_iterations}")

# API 키 유효성 검증
missing = settings.validate()
if missing:
    print(f"\n⚠ 누락된 키: {missing}")
    print("→ .env 파일을 확인하세요!")
else:
    print("\n✓ 모든 필수 API 키가 설정되어 있습니다.")

기본 프로바이더: anthropic
기본 모델:       claude-sonnet-4-5-20250929
최대 반복 횟수:  25

✓ 모든 필수 API 키가 설정되어 있습니다.


In [3]:
# (선택) .env 파일 대신 노트북에서 직접 API 키를 설정할 수도 있습니다
# 주석 해제 후 자신의 키를 입력하세요

# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["TAVILY_API_KEY"] = "tvly-..."

---
## 2. 모델 프로바이더 (Model Provider)

모델 프로바이더는 **LLM을 교체 가능하게 만드는 추상화 계층**입니다.

### 핵심 개념

- `ModelProvider`는 **추상 클래스(ABC)** 입니다
- 각 프로바이더(`AnthropicProvider`, `OpenAIProvider` 등)가 이를 구현합니다
- `get_provider("anthropic")` 팩토리 함수로 문자열 키만으로 프로바이더를 생성합니다

```
get_provider("anthropic")
  → AnthropicProvider(model_name="claude-sonnet-4-5-20250929")
    → .get_llm() → ChatAnthropic(...)  ← LangChain BaseChatModel
```

이 패턴 덕분에 **에이전트 코드를 수정하지 않고** 모델만 교체할 수 있습니다.

In [4]:
from Agent_Structure.core.model_provider import get_provider

# 팩토리 함수로 프로바이더 생성
provider = get_provider("anthropic")
print(f"프로바이더 타입: {type(provider).__name__}")
print(f"모델명:        {provider.model_name}")

프로바이더 타입: AnthropicProvider
모델명:        claude-sonnet-4-5-20250929


In [5]:
# LLM 인스턴스 획득 (LangChain BaseChatModel 호환)
llm = provider.get_llm()
print(f"LLM 타입: {type(llm).__name__}")

# LLM을 직접 호출할 수도 있습니다 (프레임워크 없이)
# response = llm.invoke("안녕하세요!")
# print(response.content)

LLM 타입: ChatAnthropic


In [6]:
from Agent_Structure.core.model_provider import _PROVIDER_MAP

# 현재 등록된 프로바이더 목록
print("등록된 프로바이더:")
for name, cls in _PROVIDER_MAP.items():
    print(f"  - {name}: {cls.__name__}")

등록된 프로바이더:
  - anthropic: AnthropicProvider
  - openai: OpenAIProvider
  - upstage: UpstageProvider


---
## 3. 도구(Tool) 레지스트리

도구 시스템은 **플러그인 패턴**으로 설계되어 있습니다.

### 작동 방식

1. `tools/` 폴더에 파이썬 파일을 만들고 `@register_tool` 데코레이터를 붙이면
2. `tools/__init__.py`에서 import할 때 **자동으로 레지스트리에 등록**됩니다
3. `build_agent()`가 레지스트리에서 도구를 수집하여 에이전트에 전달합니다

### 주요 API

| 메서드 | 설명 |
|--------|------|
| `tool_registry.get_all()` | 모든 도구 리스트 |
| `tool_registry.get_by_tag("search")` | 태그로 필터링 |
| `tool_registry.get("web_search")` | 이름으로 조회 |
| `tool_registry.list_names()` | 이름 목록 |
| `tool_registry.summary()` | 전체 요약 출력 |

In [7]:
from Agent_Structure.tools import tool_registry

# 등록된 모든 도구 요약
print(tool_registry.summary())
print(f"\n도구 이름 목록: {tool_registry.list_names()}")

=== ToolRegistry (2 tools) ===
  [search, web] web_search: Tavily API를 사용한 웹 검색.

    Args:
        query: 검색 쿼리
      
  [reasoning] think_tool: Tool for strategic reflection on research progress and decis

도구 이름 목록: ['web_search', 'think_tool']


In [8]:
# 개별 도구 정보 확인
ws = tool_registry.get("web_search")
print(f"도구명:  {ws.__name__}")
print(f"설명:\n{ws.__doc__}")

도구명:  web_search
설명:

    Tavily API를 사용한 웹 검색.

    Args:
        query: 검색 쿼리
        max_results: 최대 결과 수 (기본 5)
        topic: 검색 카테고리 — "general", "news", "finance"
        include_raw_content: 원본 HTML 포함 여부

    Returns:
        Tavily 검색 결과 딕셔너리
    


---
## 4. 첫 번째 에이전트 만들기

이제 개별 구성요소를 이해했으니, **에이전트를 조립하고 실행**해봅시다!

`run_notebook.py`는 노트북에서 편리하게 사용할 수 있는 헬퍼 함수를 제공합니다:

| 함수 | 설명 |
|------|------|
| `create_agent(**kwargs)` | `build_agent()`의 편의 래퍼. 에이전트 생성 |
| `run(agent, message)` | 동기 실행. 응답 텍스트 반환 |
| `arun(agent, message)` | 비동기 실행 (`await` 사용) |
| `stream(agent, message)` | 스트리밍. 청크 단위로 yield |

### 조립 흐름

```python
create_agent()  # == build_agent()
    ├─ settings에서 기본 프로바이더/모델 결정
    ├─ get_provider() → LLM 인스턴스
    ├─ tool_registry.get_all() → 도구 수집
    ├─ subagent_registry → 서브에이전트 수집 (설정 시)
    └─ create_deep_agent() → CompiledStateGraph 반환
```

In [9]:
from Agent_Structure.run_notebook import create_agent, run, arun, stream

# 가장 기본적인 에이전트 생성 (기본 설정 사용)
agent = create_agent()
print(f"에이전트 타입: {type(agent).__name__}")
print("에이전트가 준비되었습니다!")

에이전트 타입: CompiledStateGraph
에이전트가 준비되었습니다!


In [10]:
# 간단한 질문 실행
result = run(agent, "대한민국의 수도는 어디인가요?")
print(result)

대한민국의 수도는 **서울특별시**(Seoul)입니다.


In [37]:
from datetime import date
today = date.today()

# 도구 사용이 필요한 질문 (에이전트가 web_search를 자동으로 호출합니다)
# TAVILY_API_KEY가 설정되어 있어야 동작합니다
result = run(agent, f"오늘은 {today}야. 미국-이란 전쟁에 관련된 오늘의 주요 뉴스를 정리해주세요")
print(result)

검색 결과를 보면 3월 13-14일의 가장 최신 뉴스가 제한적입니다. 최근 며칠간의 주요 동향을 정리하겠습니다:

## 미국-이란 전쟁 주요 뉴스 (2026년 3월 14일 기준)

### **최신 동향 (3월 13-14일)**
- **미 공군 급유기 추락**: 이란 전쟁 지원 중 이라크에서 미 공군 급유기가 추락해 탑승자 6명 전원 사망
- **미시간 시나고그 공격**: 전쟁 관련 국내 안보 우려 고조

### **군사 작전 현황**
- **미국 측 발표** (3월 10일):
  - 국방장관 Pete Hegseth: "압도적으로 승리하고 있다" 선언
  - 작전 목표: ① 이란 미사일 재고 및 발사대 파괴 ② 미사일 생산능력 제거 ③ 이란 해군 파괴 ④ 핵무기 영구 저지

- **트럼프 대통령 입장** (3월 8-9일):
  - 협상 거부, "이란의 군사력과 지도부가 완전히 제거될 때까지" 전쟁 지속 시사
  - 뚜렷한 종전 시점 제시하지 않음

### **이란의 대응**
- **반격**: 이스라엘과 미군 기지가 있는 사우디아라비아, 쿠웨이트, UAE, 바레인 등 걸프 국가들 공격
- **두바이 국제공항**에 이란 드론 공격 (3월 7일)
- 사우디·쿠웨이트에서 4명 사망 (3월 8일)
- **새 최고지도자 선출** (3월 8일): 이란 국영매체 보도

### **피해 상황**
- **미군 사상자**: 최소 7명 사망 (쿠웨이트 지휘소 드론 공격 등)
- **이란 내부**: 테헤란 등 대도시 공습으로 주민 공포 확산, 인터넷 차단
- **유가 급등**: 전쟁으로 인한 경제적 영향 심화
- **이란 석유시설 타격**: 테헤란 서쪽 카라즈 등 3개 지역 연료 저장소 공격 (3월 7일)

### **전쟁 비용**
- 1주일간 전쟁 비용: **110억 달러** (약 15조 원)

### **외교적 상황**
- 이란 대통령 Pezeshkian: 주변국에 사과하며 미-이스라엘 공격에 동참하지 말 것 요청
- 미국의 항공모함 3척째 중동 배치 중

현재 전쟁은 미-이스라엘

### thread_id와 대화 메모리

`run()` 함수의 `thread_id` 파라미터로 **대화의 맥락을 유지**할 수 있습니다.  
같은 `thread_id`를 사용하면 이전 대화 내용을 기억합니다.

In [11]:
# 같은 thread_id로 대화를 이어갑니다
result1 = run(agent, "파이썬의 장점 3가지를 알려주세요", thread_id="tutorial-1")
print("[첫 번째 응답]")
print(result1)

print("\n" + "="*60 + "\n")

# 이전 대화를 기억하고 있으므로 "방금 말한 것"이 무엇인지 알 수 있습니다
result2 = run(agent, "방금 말한 것 중 첫 번째를 더 자세히 설명해주세요", thread_id="tutorial-1")
print("[이어지는 응답]")
print(result2)

[첫 번째 응답]
파이썬의 주요 장점 3가지는 다음과 같습니다:

1. **쉬운 문법과 높은 가독성**
   - 영어와 유사한 직관적인 문법으로 초보자도 쉽게 배울 수 있습니다
   - 들여쓰기로 코드 블록을 구분하여 코드가 깔끔하고 읽기 쉽습니다
   - 적은 코드로 많은 기능을 구현할 수 있어 생산성이 높습니다

2. **풍부한 라이브러리와 프레임워크**
   - 데이터 분석(Pandas, NumPy), 머신러닝(TensorFlow, PyTorch), 웹 개발(Django, Flask) 등 다양한 분야의 검증된 라이브러리가 있습니다
   - pip를 통해 수십만 개의 패키지를 쉽게 설치하고 사용할 수 있습니다
   - 표준 라이브러리도 매우 풍부하여 별도 설치 없이 많은 작업이 가능합니다

3. **다목적성과 넓은 활용 범위**
   - 웹 개발, 데이터 과학, 인공지능, 자동화, 게임 개발 등 거의 모든 분야에서 사용 가능합니다
   - 크로스 플랫폼 지원으로 Windows, Mac, Linux 등 어디서나 실행됩니다
   - 다른 언어(C/C++, Java 등)와의 연동이 용이합니다


[이어지는 응답]
**쉬운 문법과 높은 가독성**에 대해 더 자세히 설명드리겠습니다:

## 1. 영어와 유사한 직관적 문법
파이썬은 마치 영어 문장을 읽는 것처럼 코드를 작성할 수 있습니다:
```python
# 파이썬
if age >= 18:
    print("성인입니다")

# 실제로 "만약 나이가 18 이상이면, '성인입니다'를 출력하라"와 거의 동일
```

다른 언어와 비교하면:
```java
// Java - 더 복잡한 문법
if (age >= 18) {
    System.out.println("성인입니다");
}
```

## 2. 들여쓰기 기반의 깔끔한 구조
파이썬은 중괄호 `{}` 대신 들여쓰기로 코드 블록을 구분합니다:
```python
# 파이썬 - 깔끔하고 명확
for i in range(5):
    if i % 2 == 0:
        

---
## 5. 모델 교체 & 에이전트 설정

`create_agent()`에 다양한 파라미터를 전달하여 에이전트의 동작을 커스터마이징할 수 있습니다.

| 파라미터 | 설명 | 예시 |
|----------|------|------|
| `provider_name` | LLM 프로바이더 교체 | `"openai"`, `"upstage"` |
| `model_name` | 특정 모델 지정 | `"gpt-4o"`, `"claude-haiku-4-5-20251001"` |
| `system_prompt` | 에이전트 역할/성격 지정 | `"당신은 세법 전문 AI입니다."` |
| `tool_tags` | 특정 태그의 도구만 포함 | `["search"]` |
| `exclude_tools` | 특정 도구 제외 | `["web_search"]` |
| `provider` | Provider 인스턴스 직접 주입 | temperature 등 세부 설정 시 |

In [ ]:
# OpenAI 모델로 교체 (OPENAI_API_KEY 필요)
# agent_openai = create_agent(provider_name="openai", model_name="gpt-4o")
# result = run(agent_openai, "Hello! What can you do?")
# print(result)

# 더 빠른 Claude 모델 사용
# agent_fast = create_agent(model_name="claude-haiku-4-5-20251001")
# result = run(agent_fast, "빠른 응답 테스트!")
# print(result)

In [12]:
# 시스템 프롬프트로 에이전트의 역할 지정
agent_tax = create_agent(
    system_prompt="당신은 한국 세법 전문 AI 어시스턴트입니다. 모든 답변을 세법 관점에서 제공하세요."
)
result = run(agent_tax, "프리랜서 소득 신고는 어떻게 하나요?")
print(result)

# 프리랜서 소득 신고 방법

프리랜서는 **사업소득**으로 분류되며, 다음과 같이 신고합니다:

## 1. **종합소득세 신고 (매년 5월)**
- **신고기간**: 매년 5월 1일 ~ 5월 31일
- **신고대상**: 전년도(1.1~12.31) 프리랜서 소득
- **신고방법**: 
  - 홈택스(www.hometax.go.kr) 전자신고
  - 세무서 방문 신고
  - 세무대리인 통한 신고

## 2. **소득 유형별 신고**

### (1) 3.3% 원천징수된 경우
- 소득 지급 시 3.3% (소득세 3% + 지방소득세 0.3%) 원천징수
- 5월 종합소득세 신고 시 **환급 또는 추가 납부** 결정
- 사업자등록 없이도 가능

### (2) 사업자등록 후 세금계산서 발행
- 일반/간이과세자 등록
- 부가가치세 별도 신고 (1월, 7월)
- 종합소득세 신고 시 필요경비 인정받아 절세 가능

## 3. **신고 절차**
1. **자료 준비**: 소득금액증명원, 원천징수영수증, 사업 관련 경비 증빙
2. **홈택스 접속** → 종합소득세 신고
3. **소득/경비 입력**: 수입금액, 필요경비 등록
4. **세액 계산**: 자동 계산 후 확인
5. **납부 또는 환급**: 계좌이체, 카드납부 등

## 4. **절세 팁**
- **사업자등록**: 경비 인정받아 과세표준 낮춤
- **경비 증빙**: 재료비, 임차료, 통신비 등 증빙 보관
- **노란우산공제**: 소득공제 혜택
- **신용카드 사용**: 소득공제 적용

## 5. **주의사항**
- 신고 누락 시 가산세 부과 (무신고 20%, 과소신고 10%)
- 2천만원 이상 소득 시 **복식부기** 의무
- 건강보험료는 소득 기준으로 별도 산정

추가로 궁금한 사항이 있으시면 말씀해 주세요!


In [13]:
# 검색 도구만 포함하는 에이전트
agent_search_only = create_agent(tool_tags=["search"])

# 특정 도구를 제외한 에이전트
agent_no_search = create_agent(exclude_tools=["web_search"])

print("다양한 도구 설정의 에이전트를 생성할 수 있습니다.")

다양한 도구 설정의 에이전트를 생성할 수 있습니다.


In [14]:
# temperature 등 세부 설정은 provider를 직접 만들어서 전달
from Agent_Structure.core.model_provider import get_provider

# 낮은 temperature = 더 결정론적(일관된) 응답
provider_precise = get_provider("anthropic", temperature=0.1)
agent_precise = create_agent(provider=provider_precise)

# 높은 temperature = 더 창의적인 응답
provider_creative = get_provider("anthropic", temperature=0.9)
agent_creative = create_agent(provider=provider_creative)

print("temperature를 조절하여 응답 스타일을 변경할 수 있습니다.")

temperature를 조절하여 응답 스타일을 변경할 수 있습니다.


---
## 6. 스트리밍 & 비동기 실행

### 언제 어떤 방식을 사용할까?

| 방식 | 함수 | 적합한 상황 |
|------|------|------------|
| **동기** | `run()` | 노트북에서 간단히 결과만 필요할 때 |
| **스트리밍** | `stream()` | 긴 응답을 실시간으로 확인하고 싶을 때 |
| **비동기** | `arun()` | FastAPI 연동, 여러 질문 동시 처리 |

In [15]:
# 스트리밍: 응답이 생성되는 대로 실시간 출력
agent = create_agent()

print("[스트리밍 응답]")
for chunk in stream(agent, "인공지능의 역사를 간단히 설명해주세요"):
    print(chunk, end="", flush=True)
print()  # 마지막 줄바꿈

[스트리밍 응답]
[{'text': '#', 'type': 'text', 'index': 0}][{'text': ' ', 'type': 'text', 'index': 0}][{'text': '인공지능의 역사', 'type': 'text', 'index': 0}][{'text': '\n\n## 1', 'type': 'text', 'index': 0}][{'text': '. 태', 'type': 'text', 'index': 0}][{'text': '동', 'type': 'text', 'index': 0}][{'text': '기 (1', 'type': 'text', 'index': 0}][{'text': '940', 'type': 'text', 'index': 0}][{'text': '-', 'type': 'text', 'index': 0}][{'text': '1950년대', 'type': 'text', 'index': 0}][{'text': ')', 'type': 'text', 'index': 0}][{'text': '\n- **1', 'type': 'text', 'index': 0}][{'text': '943', 'type': 'text', 'index': 0}][{'text': '**:', 'type': 'text', 'index': 0}][{'text': ' 워', 'type': 'text', 'index': 0}][{'text': '런 매', 'type': 'text', 'index': 0}][{'text': '컬록', 'type': 'text', 'index': 0}][{'text': '과', 'type': 'text', 'index': 0}][{'text': ' 월', 'type': 'text', 'index': 0}][{'text': '터', 'type': 'text', 'index': 0}][{'text': ' 피', 'type': 'text', 'index': 0}][{'text': '츠가', 'type': 'text', 'index': 0}][

In [16]:
# 비동기 실행 (Jupyter 노트북에서는 await를 직접 사용할 수 있습니다)
result = await arun(agent, "비동기 실행 테스트입니다. 간단히 인사해주세요.")
print(result)

안녕하세요! 반갑습니다. 😊


In [17]:
import asyncio

# (고급) 여러 질문을 동시에 처리
questions = [
    "파이썬이란?",
    "자바스크립트란?",
    "러스트란?",
]

# 같은 에이전트를 병렬로 사용할 때는 thread_id를 다르게 설정해야 합니다
tasks = [
    arun(agent, q, thread_id=f"parallel-{i}")
    for i, q in enumerate(questions)
]

results = await asyncio.gather(*tasks)

for q, r in zip(questions, results):
    print(f"Q: {q}")
    print(f"A: {r[:150]}...")
    print()

Q: 파이썬이란?
A: 파이썬(Python)은 1991년 귀도 반 로섬(Guido van Rossum)이 개발한 고급 프로그래밍 언어입니다.

## 주요 특징

**1. 쉬운 문법**
- 읽기 쉽고 배우기 쉬운 문법
- 들여쓰기로 코드 블록을 구분
- 영어와 유사한 직관적인 문법

**2. ...

Q: 자바스크립트란?
A: 자바스크립트(JavaScript)는 웹 개발에서 가장 널리 사용되는 프로그래밍 언어입니다.

## 주요 특징

**1. 웹의 표준 언어**
- HTML, CSS와 함께 웹의 3대 핵심 기술
- 모든 주요 웹 브라우저에서 기본 지원
- 웹 페이지에 동적인 기능과 상호작용...

Q: 러스트란?
A: 러스트(Rust)는 2010년 Mozilla에서 개발한 시스템 프로그래밍 언어입니다. 주요 특징은 다음과 같습니다:

## 핵심 특징

**1. 메모리 안전성**
- 가비지 컬렉터 없이 메모리 안전성을 보장
- 소유권(ownership) 시스템으로 컴파일 타임에 메모리...



---
## 7. 프레임워크 확장 (1): 커스텀 도구 만들기

도구를 추가하는 방법은 **두 가지**입니다:

### 방법 A: 노트북 내 인라인 도구
- LangChain의 `@tool` 데코레이터 사용
- `create_agent(tools=[my_tool])`로 직접 전달
- 노트북 세션에서만 유효

### 방법 B: 파일 기반 영구 등록
- `tools/` 폴더에 파일 생성
- `@register_tool` 데코레이터 사용
- `tools/__init__.py`에 import 추가
- 프레임워크에 영구적으로 포함

In [ ]:
from langchain_core.tools import tool

# 방법 A: 노트북 내 인라인 도구 정의
@tool
def calculate(expression: str) -> str:
    """수학 계산을 수행합니다. 예: '2 + 3 * 4'

    Args:
        expression: 계산할 수식 문자열
    """
    try:
        result = eval(expression)
        return f"계산 결과: {expression} = {result}"
    except Exception as e:
        return f"계산 오류: {e}"

# 커스텀 도구를 포함하는 에이전트 생성
agent_with_calc = create_agent(tools=[calculate, ])
result = run(agent_with_calc, "123 * 456 + 789를 계산해주세요")
print(result)

**56,877**


In [19]:
from Agent_Structure.tools.base import register_tool

# 방법 B(변형): @register_tool로 글로벌 레지스트리에 등록
# (이렇게 하면 이후 create_agent() 호출 시 자동으로 포함됩니다)
@register_tool(tags=["utility", "time"])
def current_time(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """현재 시각을 반환합니다.

    Args:
        format: 시간 포맷 문자열 (기본: YYYY-MM-DD HH:MM:SS)
    """
    from datetime import datetime
    return datetime.now().strftime(format)

# 레지스트리에 등록되었는지 확인
print(tool_registry.summary())

# 이제 create_agent()가 이 도구를 자동으로 포함합니다
agent = create_agent()
result = run(agent, "지금 몇 시인가요?")
print(f"\n응답: {result}")

=== ToolRegistry (3 tools) ===
  [search, web] web_search: Tavily API를 사용한 웹 검색.

    Args:
        query: 검색 쿼리
      
  [reasoning] think_tool: Tool for strategic reflection on research progress and decis
  [utility, time] current_time: 현재 시각을 반환합니다.

    Args:
        format: 시간 포맷 문자열 (기본: YYYY

응답: 현재 시각은 2026년 3월 14일 오후 4시 47분 9초입니다.


### 파일 기반 도구 추가 방법 (영구 등록)

`tools/_template.py`를 복사하여 새 도구를 만듭니다:

1. `tools/_template.py` → `tools/my_tool.py`로 복사
2. 함수명, docstring, 로직 수정
3. `tags`를 적절히 설정 (예: `["rag", "database"]`)
4. `tools/__init__.py`에 `from . import my_tool` 추가

```python
# tools/my_tool.py 예시
from .base import register_tool

@register_tool(tags=["rag", "database"])
def qdrant_search(query: str, top_k: int = 5) -> str:
    """Qdrant 벡터DB에서 유사한 문서를 검색합니다."""
    # 여기에 실제 검색 로직 구현
    ...
```

In [21]:
# _template.py의 패턴 확인
import inspect
from Agent_Structure.tools._template import example_tool

print("=== 도구 템플릿 소스코드 ===")
print(inspect.getsource(example_tool))

=== 도구 템플릿 소스코드 ===
@register_tool(tags=["example"])
def example_tool(query: str) -> str:
    """
    예시 도구 — 이 docstring이 LLM에게 보이는 도구 설명이 됩니다.

    Args:
        query: 입력 쿼리

    Returns:
        처리 결과 문자열
    """
    # TODO: 실제 로직 구현
    return f"example_tool이 '{query}'를 처리했습니다."



---
## 8. 프레임워크 확장 (2): 서브에이전트

**서브에이전트**는 메인 에이전트가 **특정 작업을 위임**할 수 있는 전문화된 에이전트입니다.

### 왜 서브에이전트를 사용할까?

- 복잡한 작업을 **분업**으로 처리 (예: 리서치, 번역, 코드 리뷰)
- 서브에이전트는 자체 시스템 프롬프트와 도구를 가질 수 있음
- 메인 에이전트의 컨텍스트 윈도우를 오염시키지 않음

### 서브에이전트 설정 구조

```python
{
    "name": "research-agent",           # 필수: 고유 이름
    "description": "심층 리서치 전문...",  # 필수: 언제 위임할지 판단 기준
    "system_prompt": "당신은 전문 ...",   # 필수: 역할 지정
    "tools": [...],                      # 선택: 전용 도구
    "model": "..."                       # 선택: 별도 모델 (기본: 메인 에이전트와 동일)
}
```

In [23]:
import json
from Agent_Structure.subagents import subagent_registry

# 등록된 서브에이전트 확인
print(f"등록된 서브에이전트: {subagent_registry.list_names()}\n")

# research-agent 상세 설정 확인
config = subagent_registry.get("research-agent")
print(json.dumps(config, indent=2, ensure_ascii=False))

등록된 서브에이전트: ['research-agent']

{
  "name": "research-agent",
  "description": "심층 리서치가 필요한 질문을 조사합니다. 웹 검색과 자료 분석에 특화.",
  "system_prompt": "당신은 전문 리서처입니다. 주어진 주제에 대해 체계적으로 조사하고, 핵심 내용을 구조화된 형태로 정리합니다. 출처를 명확히 밝히세요.",
  "tools": []
}


In [24]:
# 서브에이전트를 포함하는 에이전트 생성
agent_with_sub = create_agent(subagent_names=["research-agent"])

# 심층 조사가 필요한 질문 → 메인 에이전트가 research-agent에 위임할 수 있음
result = run(agent_with_sub, "최근 AI 규제 동향을 조사해서 간략히 정리해주세요")
print(result)

## 최근 AI 규제 동향 정리

### 주요국 동향

**1. EU - AI Act (세계 최초 포괄적 AI 규제법)**
- **2024년 8월 1일**: 공식 발효
- **2025년 8월 2일**: 범용 AI(GPAI) 모델 규제 시작
- **2026년 8월 2일**: 고위험 AI 시스템 규제 본격화
- **2027년 8월 2일**: 전면 시행
- 위험도 기반 접근법: 금지 AI(사회적 점수, 실시간 생체인식 등) → 고위험 AI → 제한적 투명성 AI → 최소 위험 AI로 차등 규제

**2. 한국**
- **2025년 1월 21일**: 「인공지능 발전과 신뢰 기반 조성 등에 관한 기본법」 제정
- AI 시대 개인정보 규율체계 혁신 추진
- 데이터 적법 처리 근거 확대
- AI 관련 법안 13건 통과 (2016-2024 누적)

**3. 미국**
- 연방 차원 포괄적 규제법 부재
- 주정부 중심 규제 급증: 2022년 27건 → 2024년 131건
- **딥페이크 규제** 집중 (선거, 미성년자 보호 등)
- NIST AI 위험 관리 프레임워크로 가이드라인 제시

**4. 중국**
- AI 안전 거버넌스 프레임워크 2.0 발표
- 'AI 플러스' 심화 추진 정책 로드맵 발표
- 국가 통제형 모델 강화

### 글로벌 트렌드

**📈 규제 입법 급증**
- 2022년 1,062건 → 2024년 1,889건 (AI 언급 입법)
- 2016-2024년 총 204건의 AI 관련법 제정 (114개국 대상)

**🎯 주요 규제 초점**
1. **생성형 AI**: 딥페이크 오남용 방지
2. **투명성**: AI 결정 과정 설명 가능성 강화
3. **개인정보 보호**: AI 활용과 데이터 처리 균형
4. **윤리 & 책임**: AI 시스템 안전성 및 위험 관리
5. **공정경쟁**: 데이터 독점, 자사 우대 방지

**💡 기업 대응 과제**
- 내부 AI 거버넌스 체계 구축
- 위험 평가 및 관리 시스템 마련
- 투명성 및 설명 가능성 확보
- 지역별 규제 준수 체계 

In [25]:
# 커스텀 서브에이전트를 인라인으로 등록
translator_config = {
    "name": "translator-agent",
    "description": "번역이 필요한 텍스트를 전문적으로 번역합니다. 한/영/일 번역에 특화.",
    "system_prompt": (
        "당신은 전문 번역가입니다. "
        "원문의 뉘앙스와 맥락을 살려 자연스러운 번역을 제공합니다. "
        "번역 시 원문도 함께 보여주세요."
    ),
    "tools": [],
}

subagent_registry.register(translator_config)
print(f"등록된 서브에이전트: {subagent_registry.list_names()}")

# 여러 서브에이전트를 함께 사용
agent_multi = create_agent(
    subagent_names=["research-agent", "translator-agent"]
)
result = run(
    agent_multi,
    "다음 문장을 영어와 일본어로 번역해줘: '인공지능이 세상을 변화시키고 있다'"
)
print(result)

등록된 서브에이전트: ['research-agent', 'translator-agent']
번역 완료했습니다:

**영어:** Artificial intelligence is changing the world

**일본어:** 人工知能が世界を変えている (じんこうちのうがせかいをかえている)


---
## 9. 프레임워크 확장 (3): 커스텀 모델 프로바이더

`ModelProvider` 추상 클래스를 상속하고 `get_llm()`만 구현하면 됩니다.

```python
class ModelProvider(ABC):
    def __init__(self, model_name: str, **kwargs):
        self.model_name = model_name
        self.kwargs = kwargs
    
    @abstractmethod
    def get_llm(self) -> BaseChatModel:
        """LangChain 호환 LLM 인스턴스를 반환"""
        ...
```

`register_provider(name, cls)`로 런타임에 등록하면 `get_provider(name)`으로 사용할 수 있습니다.

In [26]:
from Agent_Structure.core.model_provider import ModelProvider, register_provider, _PROVIDER_MAP

class MockProvider(ModelProvider):
    """테스트용 목(Mock) 프로바이더 예시."""

    def __init__(self, model_name: str = "mock-v1", **kwargs):
        super().__init__(model_name, **kwargs)

    def get_llm(self):
        # 실제로는 LangChain BaseChatModel 호환 객체를 반환해야 합니다
        # 예: return ChatOllama(model=self.model_name)
        raise NotImplementedError("이것은 예시입니다. 실제 LLM 연결을 구현하세요.")

# 런타임에 프로바이더 등록
register_provider("mock", MockProvider)

# 등록 확인
print("등록된 프로바이더:")
for name, cls in _PROVIDER_MAP.items():
    print(f"  - {name}: {cls.__name__}")

등록된 프로바이더:
  - anthropic: AnthropicProvider
  - openai: OpenAIProvider
  - upstage: UpstageProvider
  - mock: MockProvider


### 실전 예시: Ollama 로컬 모델 연결

[Ollama](https://ollama.ai/)가 설치되어 있다면 로컬에서 무료로 LLM을 실행할 수 있습니다.

In [ ]:
# Ollama 프로바이더 예시 (Ollama 설치 + langchain-community 필요)
#
# class OllamaProvider(ModelProvider):
#     def __init__(self, model_name="llama3", **kwargs):
#         super().__init__(model_name, **kwargs)
#
#     def get_llm(self):
#         from langchain_community.chat_models import ChatOllama
#         return ChatOllama(model=self.model_name, **self.kwargs)
#
# register_provider("ollama", OllamaProvider)
# agent = create_agent(provider_name="ollama", model_name="llama3")
# result = run(agent, "안녕하세요!")
# print(result)

print("Ollama 프로바이더는 주석을 해제하고 사용하세요.")

---
## 10. 전체 아키텍처 정리

지금까지 배운 모든 구성요소가 어떻게 연결되는지 정리합니다.

```
┌──────────────────────────────────────────────────────────┐
│  config/settings.py                                      │
│  └─ Settings: API 키, 기본 프로바이더/모델               │
├──────────────────────────────────────────────────────────┤
│  core/model_provider.py                                  │
│  └─ get_provider() → AnthropicProvider / OpenAIProvider  │
│                       → .get_llm() → BaseChatModel       │
├──────────────────────────────────────────────────────────┤
│  tools/base.py            │  subagents/registry.py       │
│  └─ ToolRegistry (싱글턴) │  └─ SubagentRegistry (싱글턴)│
│     @register_tool 자동등록│     .register(config) 등록   │
├──────────────────────────────────────────────────────────┤
│  core/agent_factory.py                                   │
│  └─ build_agent(): 위 요소들을 모아                       │
│     → create_deep_agent()                                │
│     → CompiledStateGraph 반환                            │
├──────────────────────────────────────────────────────────┤
│  run_notebook.py  │  main.py (FastAPI)                   │
│  └─ run(), arun() │  └─ POST /chat                      │
│     stream()      │     GET /health                      │
└──────────────────────────────────────────────────────────┘
```

### 핵심 설계 원칙

- **관심사 분리**: 각 폴더(config, tools, subagents)는 독립적
- **단일 조립 지점**: `agent_factory.py`만 다른 모듈을 알고 있음
- **레지스트리 패턴**: 도구/서브에이전트를 추가해도 팩토리 코드 수정 불필요
- **이중 진입점**: 노트북(`run_notebook.py`)과 API(`main.py`)에서 동일한 에이전트 사용

### FastAPI 서버 실행

```bash
uvicorn Agent_Structure.main:app --reload
```

In [27]:
# 전체 조립 과정을 추적해봅시다
from Agent_Structure.config import settings
from Agent_Structure.core.model_provider import get_provider
from Agent_Structure.tools import tool_registry
from Agent_Structure.subagents import subagent_registry

print("=== Agent_Structure 조립 요약 ===\n")

print(f"1. 설정 (config/settings.py)")
print(f"   기본 프로바이더: {settings.default_provider}")
print(f"   기본 모델:      {settings.default_model}")

print(f"\n2. 모델 프로바이더 (core/model_provider.py)")
provider = get_provider(settings.default_provider)
print(f"   프로바이더: {type(provider).__name__}")
print(f"   모델명:    {provider.model_name}")

print(f"\n3. 도구 레지스트리 (tools/base.py)")
print(f"   등록 도구 수: {len(tool_registry.list_names())}")
print(f"   도구 목록:   {tool_registry.list_names()}")

print(f"\n4. 서브에이전트 레지스트리 (subagents/registry.py)")
print(f"   등록 서브에이전트: {subagent_registry.list_names()}")

print(f"\n5. 조립 (core/agent_factory.py)")
print(f"   → build_agent()가 위 요소들을 수집")
print(f"   → create_deep_agent()에 전달")
print(f"   → CompiledStateGraph (LangGraph 에이전트) 반환")

=== Agent_Structure 조립 요약 ===

1. 설정 (config/settings.py)
   기본 프로바이더: anthropic
   기본 모델:      claude-sonnet-4-5-20250929

2. 모델 프로바이더 (core/model_provider.py)
   프로바이더: AnthropicProvider
   모델명:    claude-sonnet-4-5-20250929

3. 도구 레지스트리 (tools/base.py)
   등록 도구 수: 4
   도구 목록:   ['web_search', 'think_tool', 'current_time', 'example_tool']

4. 서브에이전트 레지스트리 (subagents/registry.py)
   등록 서브에이전트: ['research-agent', 'translator-agent']

5. 조립 (core/agent_factory.py)
   → build_agent()가 위 요소들을 수집
   → create_deep_agent()에 전달
   → CompiledStateGraph (LangGraph 에이전트) 반환


---
## 11. 치트시트 & 다음 단계

### 빠른 참조

| 패턴 | 코드 |
|------|------|
| 기본 에이전트 | `create_agent()` |
| 모델 교체 | `create_agent(provider_name="openai")` |
| 특정 모델 | `create_agent(model_name="claude-haiku-4-5-20251001")` |
| 시스템 프롬프트 | `create_agent(system_prompt="당신은 ...")` |
| 도구 필터링 (태그) | `create_agent(tool_tags=["search"])` |
| 도구 제외 | `create_agent(exclude_tools=["web_search"])` |
| 커스텀 도구 추가 | `create_agent(tools=[my_tool])` |
| 서브에이전트 포함 | `create_agent(subagent_names=["research-agent"])` |
| 모든 서브에이전트 | `create_agent(include_all_subagents=True)` |
| temperature 설정 | `create_agent(provider=get_provider("anthropic", temperature=0.5))` |
| 동기 실행 | `run(agent, "질문")` |
| 비동기 실행 | `await arun(agent, "질문")` |
| 스트리밍 | `for chunk in stream(agent, "질문"): print(chunk, end="")` |
| 대화 유지 | `run(agent, "질문", thread_id="my-thread")` |

### 다음 단계

1. **나만의 도구 만들기**: `tools/_template.py`를 복사하여 시작
2. **도메인 특화 서브에이전트**: `subagents/research_agent.py`를 참고하여 생성
3. **API 서버 배포**: `main.py`로 FastAPI 서버 실행
4. **Human-in-the-Loop**: `create_agent(interrupt_on={...}, checkpointer=MemorySaver())`
5. **메모리 활성화**: `create_agent(enable_memory=True, memory_files=["knowledge.md"])`

---

**Happy Building!** 🚀